In [1]:
import tensorflow as tf
import time
import platform
import os

print("--- Verificando Configuração da GPU com TensorFlow ---")
print(f"Versão do TensorFlow: {tf.__version__}")
print(f"Versão do Python: {platform.python_version()}")
print(f"Sistema Operacional (Dentro do Container): {platform.system()} {platform.release()}")

print("\n1. Verificando Dispositivos Físicos de GPU:")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"GPU(s) Encontrada(s): {gpus}")
    for gpu in gpus:
        print(f"  - Nome: {gpu.name}")
        # Tentar obter mais detalhes (alguns detalhes podem variar de disponibilidade)
        try:
            details = tf.config.experimental.get_device_details(gpu)
            print(f"    - Tipo: {details.get('device_type', 'N/A')}")
            print(f"    - Bus ID: {details.get('physical_device_desc', 'N/A')}")
            # Note: CUDA version from driver is usually seen via nvidia-smi, not directly from tf.config
        except Exception as e:
            print(f"    - Não foi possível obter detalhes adicionais da GPU: {e}")
    print("Sucesso: GPU(s) detectada(s) pelo TensorFlow.")
else:
    print("Falha: Nenhuma GPU detectada pelo TensorFlow. Verifique a instalação e drivers.")
    print("--- FIM DO TESTE: GPU NÃO DETECTADA ---")
    exit() # Encerra o script se não houver GPU

print("\n2. Verificando Comunicação com o Driver NVIDIA (via nvidia-smi se disponível):")
try:
    # Tenta rodar nvidia-smi no terminal do container
    # Note: Este comando só funciona se nvidia-smi estiver no PATH do container,
    # o que é comum em imagens NGC.
    nvidia_smi_output = os.popen('nvidia-smi').read()
    print("Saída do nvidia-smi:")
    print(nvidia_smi_output)
    if "NVIDIA-SMI" in nvidia_smi_output:
        print("Sucesso: Comunicação com o driver NVIDIA (via nvidia-smi) estabelecida.")
        # Você pode parsear o output aqui para extrair Driver Version e CUDA Version
        driver_version_line = [line for line in nvidia_smi_output.split('\n') if 'Driver Version:' in line]
        if driver_version_line:
            print(f"  - Versão do Driver (host/WSL2): {driver_version_line[0].split('Driver Version:')[1].split(' ')[1]}")
            print(f"  - Versão CUDA (host/WSL2): {driver_version_line[0].split('CUDA Version:')[1].split(' ')[1]}")
        else:
            print("  - Não foi possível extrair a versão do driver/CUDA do nvidia-smi output.")
    else:
        print("Aviso: 'nvidia-smi' não retornou a saída esperada. Pode estar faltando ou não acessível.")
except Exception as e:
    print(f"Aviso: Não foi possível executar 'nvidia-smi': {e}")


print("\n3. Teste de Desempenho da GPU (Multiplicação de Matrizes):")
# Para garantir que o TensorFlow está usando a GPU para o cálculo.
with tf.device('/GPU:0'):
    print("Executando cálculo intensivo na GPU...")
    # Aumentar o tamanho da matriz para um teste mais robusto na 3060 Ti
    matrix_size = 8192 # Matriz 8192x8192
    a = tf.random.normal([matrix_size, matrix_size], dtype=tf.float32)
    b = tf.random.normal([matrix_size, matrix_size], dtype=tf.float32)

    # Primeira execução é geralmente mais lenta devido a inicialização
    _ = tf.matmul(a, b)

    start_time = time.time()
    c = tf.matmul(a, b)
    end_time = time.time()

    gpu_time = end_time - start_time
    print(f"Sucesso: Multiplicação de matrizes {matrix_size}x{matrix_size} na GPU: {gpu_time:.4f} segundos.")

print("\n4. Comparativo de Desempenho (GPU vs. CPU):")
# Teste na CPU para comparação
with tf.device('/CPU:0'):
    print("Executando o mesmo cálculo na CPU para comparação...")
    a_cpu = tf.random.normal([matrix_size, matrix_size], dtype=tf.float32)
    b_cpu = tf.random.normal([matrix_size, matrix_size], dtype=tf.float32)

    # Primeira execução na CPU
    _ = tf.matmul(a_cpu, b_cpu)

    start_time_cpu = time.time()
    c_cpu = tf.matmul(a_cpu, b_cpu)
    end_time_cpu = time.time()

    cpu_time = end_time_cpu - start_time_cpu
    print(f"Multiplicação de matrizes {matrix_size}x{matrix_size} na CPU: {cpu_time:.4f} segundos.")

if gpu_time < cpu_time:
    speedup = cpu_time / gpu_time
    print(f"\nVerificação Final: GPU está {speedup:.2f}x MAIS RÁPIDA que a CPU!")
    print("Parabéns, Erick! Sua GPU está acessível, configurada e rodando a todo vapor com TensorFlow!")
else:
    print("\nVerificação Final: A GPU não está mais rápida que a CPU no teste de matrizes.")
    print("Atenção: Embora a GPU esteja detectada, ela pode não estar sendo utilizada de forma otimizada.")
    print("Pode ser necessário verificar alocação de recursos do Docker Desktop ou a versão da imagem/CUDA.")

print("\n--- FIM DO TESTE ---")

2025-05-31 20:00:53.430805: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-05-31 20:00:53.448377: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-05-31 20:00:53.466685: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8473] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-05-31 20:00:53.472709: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1471] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-31 20:00:53.486985: I tensorflow/core/platform/cpu_feature_guar

--- Verificando Configuração da GPU com TensorFlow ---
Versão do TensorFlow: 2.17.0
Versão do Python: 3.12.3
Sistema Operacional (Dentro do Container): Linux 5.15.153.1-microsoft-standard-WSL2

1. Verificando Dispositivos Físicos de GPU:
GPU(s) Encontrada(s): [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
  - Nome: /physical_device:GPU:0
    - Tipo: N/A
    - Bus ID: N/A
Sucesso: GPU(s) detectada(s) pelo TensorFlow.

2. Verificando Comunicação com o Driver NVIDIA (via nvidia-smi se disponível):
Saída do nvidia-smi:
Sat May 31 20:00:57 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.57.04              Driver Version: 576.52         CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memor

I0000 00:00:1748721656.921332     271 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1748721657.061207     271 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1748721657.061246     271 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1748721657.062146     271 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1748721657.216951     271 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


Executando cálculo intensivo na GPU...
Sucesso: Multiplicação de matrizes 8192x8192 na GPU: 0.0003 segundos.

4. Comparativo de Desempenho (GPU vs. CPU):
Executando o mesmo cálculo na CPU para comparação...


I0000 00:00:1748721657.217022     271 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1748721657.217036     271 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1748721657.330030     271 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
I0000 00:00:1748721657.330080     271 cuda_executor.cc:1001] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2025-05-31 20:00:57.330089: I tensorflow/core/common_runtime/gpu/gpu_device.cc:2112] Could not identify NUMA node of platform GPU id 0, defaulting to 0.  Your kernel may not have been built with NUMA support.

Multiplicação de matrizes 8192x8192 na CPU: 2.0819 segundos.

Verificação Final: GPU está 6492.28x MAIS RÁPIDA que a CPU!
Parabéns, Erick! Sua GPU está acessível, configurada e rodando a todo vapor com TensorFlow!

--- FIM DO TESTE ---
